In [6]:
import sqlite3
import os

def inspect_scenarios_table(db_path):
    """
    Connects to a SQLite database and inspects the 'scenarios' table.

    It prints the schema and the first 5 rows of the 'scenarios' table if it exists.

    Args:
        db_path (str): The full path to the SQLite database file.
    """
    target_table = 'scenario'

    if not os.path.exists(db_path):
        print(f"Error: Database file not found at '{db_path}'")
        print("Please update the 'db_path' variable with the correct location.")
        return

    conn = None  # Initialize conn to None
    try:
        # Connect to the SQLite database
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()

        print(f"--- Successfully connected to {os.path.basename(db_path)} ---")

        # --- 1. Check if the 'scenarios' table exists ---
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name=?;", (target_table,))
        table_exists = cursor.fetchone()

        if not table_exists:
            print(f"\nError: Table '{target_table}' not found in the database.")
            # List all available tables for context
            cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
            all_tables = cursor.fetchall()
            if all_tables:
                print("Available tables are:")
                for table in all_tables:
                    print(f"  - {table[0]}")
            return

        # --- 2. Print the schema for the 'scenarios' table ---
        print(f"\n--- Schema for table: {target_table} ---")
        cursor.execute(f"PRAGMA table_info({target_table});")
        columns = cursor.fetchall()
        for column in columns:
            # Column info is a tuple: (id, name, type, notnull, default_value, pk)
            print(f"  - Column: {column[1]} (Type: {column[2]})")

        # --- 3. View the first 5 rows of the 'scenarios' table ---
        print(f"\n--- First 5 rows of '{target_table}' ---")
        cursor.execute(f"SELECT * FROM {target_table} LIMIT 5;")
        rows = cursor.fetchall()
        if not rows:
            print("  - Table is empty.")
        else:
            # Print header
            header = [description[0] for description in cursor.description]
            print(f"  {header}")
            # Print rows
            for row in rows:
                print(f"  - {row}")

    except sqlite3.Error as e:
        print(f"Database error: {e}")
    finally:
        # Close the connection
        if conn:
            conn.close()
            print("\n--- Connection closed. ---")




In [7]:
db_path = '../app/databases/nodes.db'   
database_path = 'path/to/your/scenarios.db'  # <--- CHANGE THIS PATH

inspect_scenarios_table(db_path)


--- Successfully connected to nodes.db ---

Error: Table 'scenario' not found in the database.
Available tables are:
  - nodes

--- Connection closed. ---


In [8]:
import sqlite3
import os

# --- IMPORTANT: Change this to the actual path of your database file ---
db_path = '../app/databases/scenarios.db'

if not os.path.exists(db_path):
    print(f"Error: Database file not found at '{db_path}'")
    print("Please update the 'db_path' variable with the correct location.")
else:
    try:
        # Connect to the SQLite database
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()

        print(f"--- Successfully connected to {os.path.basename(db_path)} ---")

        # --- 1. List all tables in the database ---
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
        tables = cursor.fetchall()
        print("\nTables found in the database:")
        if not tables:
            print("  - No tables found.")
        else:
            for table_name in tables:
                print(f"  - {table_name[0]}")

        # --- 2. Print the schema for each table ---
        if tables:
            print("\nSchema for each table:")
            for table_name in tables:
                table_name = table_name[0]
                print(f"\n--- Table: {table_name} ---")
                cursor.execute(f"PRAGMA table_info({table_name});")
                columns = cursor.fetchall()
                for column in columns:
                    # Column info is a tuple: (id, name, type, notnull, default_value, pk)
                    print(f"  - Column: {column[1]} ({column[2]})")

        # --- 3. (Optional) View the first 5 rows of a table ---
        # Replace 'your_table_name' with a table you found in step 1
        if tables:
            target_table = tables[0][0] # Just using the first table as an example
            print(f"\n--- First 5 rows of '{target_table}' ---")
            try:
                cursor.execute(f"SELECT * FROM {target_table} LIMIT 5;")
                rows = cursor.fetchall()
                if not rows:
                    print("  - Table is empty.")
                else:
                    for row in rows:
                        print(f"  - {row}")
            except sqlite3.Error as e:
                print(f"Could not query table '{target_table}': {e}")


    except sqlite3.Error as e:
        print(f"Database error: {e}")
    finally:
        # Close the connection
        if conn:
            conn.close()
            print("\n--- Connection closed. ---")

--- Successfully connected to scenarios.db ---

Tables found in the database:
  - scenarios

Schema for each table:

--- Table: scenarios ---
  - Column: name (TEXT)
  - Column: start_time (TEXT)
  - Column: end_time (TEXT)
  - Column: title (TEXT)
  - Column: description (TEXT)
  - Column: deployment (TEXT)
  - Column: federation (TEXT)
  - Column: topology (TEXT)
  - Column: nodes (TEXT)
  - Column: nodes_graph (TEXT)
  - Column: n_nodes (TEXT)
  - Column: matrix (TEXT)
  - Column: random_topology_probability (TEXT)
  - Column: dataset (TEXT)
  - Column: iid (TEXT)
  - Column: partition_selection (TEXT)
  - Column: partition_parameter (TEXT)
  - Column: model (TEXT)
  - Column: agg_algorithm (TEXT)
  - Column: rounds (TEXT)
  - Column: logginglevel (TEXT)
  - Column: report_status_data_queue (TEXT)
  - Column: accelerator (TEXT)
  - Column: network_subnet (TEXT)
  - Column: network_gateway (TEXT)
  - Column: epochs (TEXT)
  - Column: attack_params (TEXT)
  - Column: reputation (TEXT)

In [9]:
import sqlite3
import os

def get_nebula_to_title_mapping(db_path):
    """
    Connects to the scenarios.db and retrieves a mapping of nebula_id to title.

    Args:
        db_path (str): The full path to the scenarios.db SQLite database file.

    Returns:
        dict: A dictionary where keys are the nebula_ids (names) and values
              are the corresponding titles. Returns an empty dictionary on error.
    """
    if not os.path.exists(db_path):
        print(f"Error: Database file not found at '{db_path}'")
        return {}

    mapping = {}
    conn = None
    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()

        # SQL query to select only the 'name' and 'title' columns
        cursor.execute("SELECT name, title FROM scenarios")
        rows = cursor.fetchall()

        # Create a dictionary from the results
        for row in rows:
            nebula_id, title = row
            mapping[nebula_id] = title

    except sqlite3.Error as e:
        print(f"Database error: {e}")
    finally:
        if conn:
            conn.close()

    return mapping

if __name__ == '__main__':
    # --- USAGE EXAMPLE ---
    # IMPORTANT: Change this to the actual path of your database file.
    # It should point to '.../config/databases/scenarios.db'
    database_path = '../app/databases/scenarios.db'  # <--- CHANGE THIS PATH

    # Get the mapping from the database
    nebula_title_map = get_nebula_to_title_mapping(database_path)

    if not nebula_title_map:
        print("\nCould not retrieve the mapping. Please check the path and database integrity.")
    else:
        print("--- Nebula ID to Title Mapping ---")
        for nebula_id, title in nebula_title_map.items():
            print(f"'{nebula_id}': '{title}'")
        print("\n--- End of Mapping ---")

--- Nebula ID to Title Mapping ---
'nebula_DFL_2025_10_14_19_47_33': 'vantest'
'nebula_DFL_2025_10_14_19_55_23': 'van2'
'nebula_DFL_2025_10_14_19_56_49': 'van2'
'nebula_DFL_2025_10_14_19_58_35': 'vantest3'
'nebula_DFL_2025_10_14_20_13_28': 'est3'
'nebula_DFL_2025_10_14_20_24_37': 'test4'
'nebula_DFL_2025_10_14_20_28_15': 'tessss'
'nebula_DFL_2025_10_14_20_50_29': 'vantest'
'nebula_DFL_2025_10_14_20_55_18': 'test '
'nebula_DFL_2025_10_14_21_01_58': 'test4'
'nebula_DFL_2025_10_14_21_11_57': 'test4'
'nebula_DFL_2025_10_14_21_18_37': 'test4'
'nebula_DFL_2025_10_14_21_34_16': 'test4'
'nebula_DFL_2025_10_15_20_43_36': 'test'

--- End of Mapping ---
